In [12]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd


In [13]:
df = pd.read_csv("../data/processed/demand_features.csv", parse_dates=["date"])
df.head()

,date,sku_id,units_sold,year,month,day,day_of_week,day_of_year,is_weekend,is_back_to_school,is_holiday_season,lag_1,lag_7,lag_14,rolling_mean_7,rolling_mean_30
0,2022-01-31,MBA13,305,2022,1,31,0,31,0,0,0,274.0,301.0,274.0,284.428571,283.933333
1,2022-02-01,MBA13,149,2022,2,1,1,32,0,0,0,305.0,319.0,326.0,285.000000,285.600000
2,2022-02-02,MBA13,314,2022,2,2,2,33,0,0,0,149.0,300.0,260.0,260.714286,282.800000
3,2022-02-03,MBA13,324,2022,2,3,3,34,0,0,0,314.0,290.0,454.0,262.714286,283.733333
4,2022-02-04,MBA13,287,2022,2,4,4,35,0,0,0,324.0,284.0,322.0,267.571429,285.000000


In [23]:
feature_cols = [col for col in df.columns if col not in ["date", "sku_id", "units_sold"]]
# print("Feature columns:", feature_cols)
# models to compare
model_configs = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1, max_iter = 100000),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter = 100000),
}

all_results = []
trained_models = {}  # trained_models[model_name][sku_id] = model object

for model_name, model_template in model_configs.items():
    trained_models[model_name] = {}
    
    for sku in df["sku_id"].unique():
        sku_df = df[df["sku_id"] == sku].sort_values("date").reset_index(drop=True)
        
        split_idx = int(len(sku_df) * 0.8)
        train = sku_df.iloc[:split_idx]
        test = sku_df.iloc[split_idx:]
        
        X_train, y_train = train[feature_cols], train["units_sold"]
        X_test, y_test = test[feature_cols], test["units_sold"]
        
        model = model_template.__class__(**model_template.get_params())  # fresh copy
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
        r2 = r2_score(y_test, preds)
        
        all_results.append({
            "model": model_name,
            "sku_id": sku,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2
        })
        trained_models[model_name][sku] = model

results_df = pd.DataFrame(all_results)
results_df

,model,sku_id,RMSE,MAE,MAPE,R2
0,LinearRegression,MBA13,60.264608,39.266877,13.780328,0.590264
1,LinearRegression,MBA15,67.341768,42.038999,16.806832,0.459958
2,LinearRegression,MBP14,45.601841,28.809173,15.911792,0.494132
3,LinearRegression,MBP16,25.039950,15.096356,10.263915,0.544335
4,Ridge,MBA13,60.287072,39.257319,13.756033,0.589958
5,Ridge,MBA15,67.270713,41.750599,16.825197,0.461097
6,Ridge,MBP14,45.558173,28.791234,15.900009,0.495100
7,Ridge,MBP16,24.981258,15.037150,10.290116,0.546468
8,Lasso,MBA13,60.271310,39.232169,13.736914,0.590173
9,Lasso,MBA15,67.221548,41.648558,16.818267,0.461884


In [15]:
summary_df = results_df.groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)
summary_df = summary_df.sort_values("RMSE")
summary_df

,RMSE,MAE,MAPE,R2
model,,,,
Lasso,49.509,31.177,14.188,0.523
Ridge,49.524,31.209,14.193,0.523
LinearRegression,49.562,31.303,14.191,0.522
ElasticNet,52.323,34.077,15.469,0.467


In [16]:
results_df.to_csv("../data/processed/baseline_regression_results.csv", index=False)
summary_df.to_csv("../data/processed/baseline_regression_summary.csv")
print("Saved baseline results.")

Saved baseline results.


In [17]:
print(df.groupby('sku_id')['units_sold'].mean())

sku_id
MBA13    335.580713
MBA15    286.540338
MBP14    216.808630
MBP16    133.564728
Name: units_sold, dtype: float64


Updated:

In [18]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

df1 = pd.read_csv("../data/processed/demand_timeseries_v2.csv", parse_dates=["date"])
df1.head()

,date,sku_id,units_sold,year,month,day,day_of_week,day_of_year,is_weekend,is_back_to_school,is_holiday_season,lag_1,lag_7,lag_14,rolling_mean_7,rolling_mean_30
0,2022-01-31,MBA13,305,2022,1,31,0,31,0,0,0,274.0,301.0,274.0,284.428571,283.933333
1,2022-02-01,MBA13,149,2022,2,1,1,32,0,0,0,305.0,319.0,326.0,285.000000,285.600000
2,2022-02-02,MBA13,314,2022,2,2,2,33,0,0,0,149.0,300.0,260.0,260.714286,282.800000
3,2022-02-03,MBA13,324,2022,2,3,3,34,0,0,0,314.0,290.0,454.0,262.714286,283.733333
4,2022-02-04,MBA13,287,2022,2,4,4,35,0,0,0,324.0,284.0,322.0,267.571429,285.000000


In [19]:
feature_cols1 = [col for col in df1.columns if col not in ["date", "sku_id", "units_sold"]]
# print("Feature columns:", feature_cols1)
# models to compare
model_configs1 = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1, max_iter = 100000),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter = 100000),
}

all_results1 = []
trained_models1 = {}  # trained_models[model_name][sku_id] = model object

for model_name, model_template in model_configs1.items():
    trained_models1[model_name] = {}
    
    for sku in df1["sku_id"].unique():
        sku_df = df1[df1["sku_id"] == sku].sort_values("date").reset_index(drop=True)
        
        split_idx = int(len(sku_df) * 0.8)
        train = sku_df.iloc[:split_idx]
        test = sku_df.iloc[split_idx:]
        
        X_train, y_train = train[feature_cols1], train["units_sold"]
        X_test, y_test = test[feature_cols1], test["units_sold"]
        
        model = model_template.__class__(**model_template.get_params())  # fresh copy
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        rmse1 = np.sqrt(mean_squared_error(y_test, preds))
        mae1 = mean_absolute_error(y_test, preds)
        mape1 = np.mean(np.abs((y_test - preds) / y_test)) * 100
        r21 = r2_score(y_test, preds)
        
        all_results1.append({
            "model": model_name,
            "sku_id": sku,
            "RMSE": rmse1,
            "MAE": mae1,
            "MAPE": mape1,
            "R2": r21
        })
        trained_models1[model_name][sku] = model

results_df1 = pd.DataFrame(all_results1)
results_df1

,model,sku_id,RMSE,MAE,MAPE,R2
0,LinearRegression,MBA13,60.264608,39.266877,13.780328,0.590264
1,LinearRegression,MBA15,67.341768,42.038999,16.806832,0.459958
2,LinearRegression,MBP14,45.601841,28.809173,15.911792,0.494132
3,LinearRegression,MBP16,25.039950,15.096356,10.263915,0.544335
4,Ridge,MBA13,60.287072,39.257319,13.756033,0.589958
5,Ridge,MBA15,67.270713,41.750599,16.825197,0.461097
6,Ridge,MBP14,45.558173,28.791234,15.900009,0.495100
7,Ridge,MBP16,24.981258,15.037150,10.290116,0.546468
8,Lasso,MBA13,60.271310,39.232169,13.736914,0.590173
9,Lasso,MBA15,67.221548,41.648558,16.818267,0.461884


In [20]:
summary_df1 = results_df1.groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)
summary_df1 = summary_df1.sort_values("RMSE")
summary_df1

,RMSE,MAE,MAPE,R2
model,,,,
Lasso,49.509,31.177,14.188,0.523
Ridge,49.524,31.209,14.193,0.523
LinearRegression,49.562,31.303,14.191,0.522
ElasticNet,52.323,34.077,15.469,0.467


In [21]:
results_df1.to_csv("../data/processed/baseline_regression_results_v2.csv", index=False)
summary_df1.to_csv("../data/processed/baseline_regression_summary_v2.csv", index=False)
print("Saved baseline results.")

Saved baseline results.


In [25]:
print(df1.groupby('sku_id')['units_sold'].mean())

sku_id
MBA13    335.580713
MBA15    286.540338
MBP14    216.808630
MBP16    133.564728
Name: units_sold, dtype: float64


In [29]:
# Load both datasets
df = pd.read_csv("../data/processed/demand_features.csv", parse_dates=["date"])
df1 = pd.read_csv("../data/processed/demand_features_v2.csv", parse_dates=["date"])

# 1. Check SKU overlap
skus_df = set(df["sku_id"].unique())
skus_df1 = set(df1["sku_id"].unique())

print("SKUs only in df:", skus_df - skus_df1)
print("SKUs only in df1:", skus_df1 - skus_df)
print("Common SKUs:", len(skus_df & skus_df1), "/", len(skus_df | skus_df1))

# 2. Check date range overlap (overall)
print("\ndf date range:", df["date"].min(), "to", df["date"].max())
print("df1 date range:", df1["date"].min(), "to", df1["date"].max())

# 3. Check overlap per SKU (date, sku_id) pairs — this catches leakage between train/test sources
keys_df = set(zip(df["sku_id"], df["date"]))
keys_df1 = set(zip(df1["sku_id"], df1["date"]))

overlap = keys_df & keys_df1
print(f"\nOverlapping (sku_id, date) rows: {len(overlap)} out of {len(keys_df)} in df and {len(keys_df1)} in df1")

if overlap:
    overlap_df = pd.DataFrame(list(overlap), columns=["sku_id", "date"]).sort_values(["sku_id", "date"])
    print(overlap_df.head(20))

SKUs only in df: set()
SKUs only in df1: set()
Common SKUs: 4 / 4

df date range: 2022-01-31 00:00:00 to 2025-12-31 00:00:00
df1 date range: 2022-01-31 00:00:00 to 2025-12-31 00:00:00

Overlapping (sku_id, date) rows: 4629 out of 4629 in df and 4629 in df1
     sku_id       date
137   MBA13 2022-01-31
4568  MBA13 2022-02-01
4572  MBA13 2022-02-02
2422  MBA13 2022-02-03
796   MBA13 2022-02-04
1814  MBA13 2022-02-05
481   MBA13 2022-02-06
2471  MBA13 2022-02-07
2495  MBA13 2022-02-08
54    MBA13 2022-02-09
1873  MBA13 2022-02-10
1867  MBA13 2022-02-11
4198  MBA13 2022-02-12
785   MBA13 2022-02-13
2076  MBA13 2022-02-14
776   MBA13 2022-02-15
4146  MBA13 2022-02-16
3519  MBA13 2022-02-17
1733  MBA13 2022-02-18
2197  MBA13 2022-02-19


In [27]:
merged = df.merge(df1, on=["sku_id", "date"], suffixes=("_v1", "_v2"))
diffs = merged[merged["units_sold_v1"] != merged["units_sold_v2"]]
print(f"{len(diffs)} rows where units_sold differs between df and df1")

0 rows where units_sold differs between df and df1


updated_v3

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load features
df_lin_v3 = pd.read_csv("../data/processed/demand_features_v3.csv", parse_dates=["date"])

feature_cols_v3 = ['year', 'month', 'day_of_week', 'is_weekend', 'is_back_to_school', 'is_holiday_season', 
                   'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_mean_30', 'is_promo', 'is_stockout']

lin_models_v3 = {
    "LinearRegression_v3": LinearRegression(),
    "Ridge_v3": Ridge(alpha=1.0),
    "Lasso_v3": Lasso(alpha=0.1, max_iter=100000),
    "ElasticNet_v3": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=100000)
}

lin_results_v3 = []

for model_name, model_template in lin_models_v3.items():
    for sku in df_lin_v3["sku_id"].unique():
        sku_df = df_lin_v3[df_lin_v3["sku_id"] == sku].reset_index(drop=True)
        split_idx = int(len(sku_df) * 0.8)
        
        train, test = sku_df.iloc[:split_idx], sku_df.iloc[split_idx:]
        
        X_train, y_train_diff = train[feature_cols_v3], train["units_sold_diff"]
        X_test, y_test_actual, test_lag_1 = test[feature_cols_v3], test["units_sold"], test["lag_1"]
        
        model = model_template.__class__(**model_template.get_params())
        model.fit(X_train, y_train_diff)
        
        # Reconstruct actual predictions
        preds_actual = test_lag_1 + model.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test_actual, preds_actual))
        mae = mean_absolute_error(y_test_actual, preds_actual)
        mape = np.mean(np.abs((y_test_actual - preds_actual) / y_test_actual)) * 100
        r2 = r2_score(y_test_actual, preds_actual)
        
        lin_results_v3.append({"model": model_name, "sku_id": sku, "RMSE": rmse, "MAE": mae, "MAPE": mape, "R2": r2})

res_lin_df_v3 = pd.DataFrame(lin_results_v3)
summary_lin_v3 = res_lin_df_v3.groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3).sort_values("RMSE")

print("--- Linear Baseline (v3) ---")
print(summary_lin_v3)
res_lin_df_v3.to_csv("../data/processed/baseline_regression_results_v3.csv", index=False)
summary_lin_v3.to_csv("../data/processed/baseline_regression_summary_v3.csv")

--- Linear Baseline (v3) ---
                       RMSE     MAE    MAPE     R2
model                                             
LinearRegression_v3  54.926  29.769  12.275  0.547
Ridge_v3             55.004  29.709  12.249  0.546
Lasso_v3             55.035  29.819  12.299  0.545
ElasticNet_v3        61.402  34.368  14.599  0.437
